1. Instalasi dan cek GPU

In [ ]:
!pip -q install thop torchinfo scikit-learn

import torch
import torchvision

print("PyTorch:", torch.__version__)
print("Torchvision:", torchvision.__version__)
print("GPU tersedia:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    device = torch.device("cuda")
else:
    raise RuntimeError(
        "GPU belum aktif. Pilih Runtime > Change runtime type > T4 GPU."
    )

PyTorch: 2.11.0+cu130
Torchvision: 0.26.0+cu130
GPU tersedia: True
GPU: Tesla T4


2. Unduh dan ekstrak dataset Imagenette 160 px

In [ ]:

from pathlib import Path
import urllib.request
import tarfile

DATA_DIR = Path("/content/imagenette2-160")
ARCHIVE = Path("/content/imagenette2-160.tgz")
URL = "https://s3.amazonaws.com/fast-ai-imageclas/imagenette2-160.tgz"

if not DATA_DIR.exists():
    if not ARCHIVE.exists() or ARCHIVE.stat().st_size == 0:
        print("Mengunduh Imagenette 160 px...")
        urllib.request.urlretrieve(URL, ARCHIVE)

    if not tarfile.is_tarfile(ARCHIVE):
        raise RuntimeError(
            "Arsip tidak valid. Hapus arsip yang gagal dan unduh ulang."
        )

    print("Mengekstrak dataset...")
    with tarfile.open(ARCHIVE, "r:gz") as tar:
        tar.extractall("/content", filter="data")

print("Folder dataset:", DATA_DIR)
print("Folder tersedia:", DATA_DIR.exists())
print("Folder:", [p.name for p in DATA_DIR.iterdir()])


Folder dataset: /content/imagenette2-160
Folder tersedia: True
Folder: ['val', 'noisy_imagenette.csv', 'train', '.DS_Store']


3. Muat dataset dan buat validasi internal

In [ ]:

from torchvision import datasets, transforms
from torch.utils.data import DataLoader, Subset
from sklearn.model_selection import train_test_split

IMAGE_SIZE = 160
BATCH_SIZE = 64
SEED = 0

train_transform = transforms.Compose([
    transforms.RandomResizedCrop(IMAGE_SIZE),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

eval_transform = transforms.Compose([
    transforms.Resize(IMAGE_SIZE),
    transforms.CenterCrop(IMAGE_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

# Dua salinan dataset train agar transformasi train dan validasi terpisah
train_aug = datasets.ImageFolder(
    DATA_DIR / "train", transform=train_transform
)
train_eval = datasets.ImageFolder(
    DATA_DIR / "train", transform=eval_transform
)
test_dataset = datasets.ImageFolder(
    DATA_DIR / "val", transform=eval_transform
)

assert train_aug.classes == test_dataset.classes
assert len(train_aug.classes) == 10

indices = list(range(len(train_aug)))
labels = train_aug.targets

train_idx, internal_val_idx = train_test_split(
    indices,
    test_size=0.10,
    random_state=SEED,
    stratify=labels
)

train_dataset = Subset(train_aug, train_idx)
internal_val_dataset = Subset(train_eval, internal_val_idx)

train_loader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE,
    shuffle=True, num_workers=2, pin_memory=True
)
internal_val_loader = DataLoader(
    internal_val_dataset, batch_size=BATCH_SIZE,
    shuffle=False, num_workers=2, pin_memory=True
)
test_loader = DataLoader(
    test_dataset, batch_size=BATCH_SIZE,
    shuffle=False, num_workers=2, pin_memory=True
)

print("Kelas:", train_aug.classes)
print("Data training:", len(train_dataset))
print("Validasi internal:", len(internal_val_dataset))
print("Evaluasi akhir:", len(test_dataset))


Kelas: ['n01440764', 'n02102040', 'n02979186', 'n03000684', 'n03028079', 'n03394916', 'n03417042', 'n03425413', 'n03445777', 'n03888257']
Data training: 8522
Validasi internal: 947
Evaluasi akhir: 3925


4. Definisikan model ConvNeXt-Tiny

In [ ]:

import torch.nn as nn
from torchvision.models import convnext_tiny

NUM_CLASSES = 10
device = torch.device("cuda")

class ConvNeXtTiny(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.model = convnext_tiny(weights=None)
        in_features = self.model.classifier[2].in_features
        self.model.classifier[2] = nn.Linear(
            in_features, num_classes
        )

    def forward(self, x):
        return self.model(x)

model = ConvNeXtTiny(NUM_CLASSES).to(device)

# Verifikasi dimensi
with torch.no_grad():
    dummy = torch.randn(1, 3, 160, 160, device=device)
    output = model(dummy)

print("Input:", tuple(dummy.shape))
print("Output:", tuple(output.shape))


Input: (1, 3, 160, 160)
Output: (1, 10)


5. Ukur parameter, MACs, dan ringkasan arsitektur

In [ ]:
from pathlib import Path
from thop import profile
from torchinfo import summary

model.eval()

# Hitung parameter
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(
    p.numel() for p in model.parameters() if p.requires_grad
)

# Hitung MACs
dummy = torch.randn(
    1, 3, IMAGE_SIZE, IMAGE_SIZE, device=device
)

with torch.no_grad():
    macs, _ = profile(model, inputs=(dummy,), verbose=False)

print("=" * 70)
print("INFORMASI MODEL CONVNEXT-TINY")
print("=" * 70)
print(f"Total parameter       : {total_params:,}")
print(f"Parameter (juta)      : {total_params / 1e6:.3f}")
print(f"Trainable parameter   : {trainable_params:,}")
print(f"Non-trainable param   : {total_params - trainable_params:,}")
print(f"MACs                  : {macs / 1e9:.4f} GMACs")
print(f"Ukuran input          : (1, 3, {IMAGE_SIZE}, {IMAGE_SIZE})")
print(f"Jumlah kelas output   : {NUM_CLASSES}")

# Ringkasan arsitektur detail
model_summary = summary(
    model,
    input_size=(1, 3, IMAGE_SIZE, IMAGE_SIZE),
    device=str(device),
    col_names=[
        "input_size",
        "output_size",
        "num_params",
        "trainable",
    ],
    depth=4,
    verbose=0,
)

print(model_summary)

# Simpan ringkasan ke file
OUTPUT_DIR = Path("/content/outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SUMMARY_PATH = OUTPUT_DIR / "convnext_tiny_model_summary.txt"

with open(SUMMARY_PATH, "w", encoding="utf-8") as file:
    file.write("RINGKASAN MODEL CONVNEXT-TINY\n")
    file.write("=" * 70 + "\n")
    file.write(f"Total parameter       : {total_params:,}\n")
    file.write(f"Parameter (juta)      : {total_params / 1e6:.3f}\n")
    file.write(f"Trainable parameter   : {trainable_params:,}\n")
    file.write(f"Non-trainable param   : {total_params - trainable_params:,}\n")
    file.write(f"MACs                  : {macs / 1e9:.4f} GMACs\n")
    file.write(f"Ukuran input          : (1, 3, {IMAGE_SIZE}, {IMAGE_SIZE})\n")
    file.write(f"Jumlah kelas output   : {NUM_CLASSES}\n\n")
    file.write(str(model_summary))
    file.write("\n")

print("\nRingkasan berhasil disimpan.")
print("Lokasi:", SUMMARY_PATH)
print("Ukuran file:", f"{SUMMARY_PATH.stat().st_size:,} bytes")

INFORMASI MODEL CONVNEXT-TINY
Total parameter       : 27,827,818
Parameter (juta)      : 27.828
Trainable parameter   : 27,827,818
Non-trainable param   : 0
MACs                  : 2.2772 GMACs
Ukuran input          : (1, 3, 160, 160)
Jumlah kelas output   : 10
Layer (type:depth-idx)                             Input Shape               Output Shape              Param #                   Trainable
ConvNeXtTiny                                       [1, 3, 160, 160]          [1, 10]                   --                        True
├─ConvNeXt: 1-1                                    [1, 3, 160, 160]          [1, 10]                   --                        True
│    └─Sequential: 2-1                             [1, 3, 160, 160]          [1, 768, 5, 5]            --                        True
│    │    └─Conv2dNormActivation: 3-1              [1, 3, 160, 160]          [1, 96, 40, 40]           --                        True
│    │    │    └─Conv2d: 4-1                       [1, 3, 160, 

6. Uji pelatihan satu epoch dan ukur waktu serta memori

In [ ]:

import time
import torch.optim as optim

criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(
    model.parameters(),
    lr=0.05,
    momentum=0.9,
    weight_decay=5e-4
)

torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()
model.train()

start = time.perf_counter()
running_loss = 0.0
correct = 0
total = 0

for images, labels in train_loader:
    images = images.to(device, non_blocking=True)
    labels = labels.to(device, non_blocking=True)

    optimizer.zero_grad(set_to_none=True)
    outputs = model(images)
    loss = criterion(outputs, labels)
    loss.backward()
    optimizer.step()

    running_loss += loss.item() * labels.size(0)
    correct += (outputs.argmax(1) == labels).sum().item()
    total += labels.size(0)

torch.cuda.synchronize()
epoch_seconds = time.perf_counter() - start

peak_memory_gb = torch.cuda.max_memory_allocated() / 1024**3

print(f"Durasi training epoch: {epoch_seconds / 60:.2f} menit")
print(f"Training loss: {running_loss / total:.4f}")
print(f"Training accuracy: {100 * correct / total:.2f}%")
print(f"Peak allocated GPU memory: {peak_memory_gb:.3f} GB")
print("GPU:", torch.cuda.get_device_name(0))


Durasi training epoch: 1.15 menit
Training loss: 4.8250
Training accuracy: 12.29%
Peak allocated GPU memory: 3.826 GB
GPU: Tesla T4


7. Validasi internal

In [ ]:
model.eval()

val_loss = 0.0
val_correct = 0
val_total = 0

start = time.perf_counter()

with torch.no_grad():
    for images, labels in internal_val_loader:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        outputs = model(images)
        loss = criterion(outputs, labels)

        val_loss += loss.item() * labels.size(0)
        val_correct += (
            outputs.argmax(1) == labels
        ).sum().item()
        val_total += labels.size(0)

torch.cuda.synchronize()
val_seconds = time.perf_counter() - start

print(f"Validation loss: {val_loss / val_total:.4f}")
print(f"Validation accuracy: {100 * val_correct / val_total:.2f}%")
print(f"Waktu validasi: {val_seconds / 60:.2f} menit")

Validation loss: 2.2621
Validation accuracy: 17.32%
Waktu validasi: 0.04 menit


8. Estimasi total 3 run × 30 epoch

In [ ]:

EPOCHS = 30
RUNS = 3

estimated_epoch_seconds = epoch_seconds + val_seconds
estimated_total_hours = (
    estimated_epoch_seconds * EPOCHS * RUNS / 3600
)

print(f"Estimasi waktu per epoch: {estimated_epoch_seconds / 60:.2f} menit")
print(f"Estimasi total 3 run x 30 epoch: {estimated_total_hours:.2f} jam")
print("Estimasi belum mencakup startup, penyimpanan checkpoint, dan overhead tambahan.")


Estimasi waktu per epoch: 1.19 menit
Estimasi total 3 run x 30 epoch: 1.79 jam
Estimasi belum mencakup startup, penyimpanan checkpoint, dan overhead tambahan.


9. Checkpoint

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

CHECKPOINT_DIR = Path(
    "/content/drive/MyDrive/ConvNeXt_Project/checkpoints"
)
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

checkpoint_path = CHECKPOINT_DIR / "convnext_tiny_benchmark.pth"

torch.save(
    {
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "classes": train_aug.classes,
        "image_size": IMAGE_SIZE,
        "num_classes": NUM_CLASSES,
        "epoch": 1,
        "benchmark_only": True,
    },
    checkpoint_path,
)

print("Checkpoint tersimpan di:", checkpoint_path)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Checkpoint tersimpan di: /content/drive/MyDrive/ConvNeXt_Project/checkpoints/convnext_tiny_benchmark.pth
